# Setup env

In [2]:
import pandas as pd
from pathlib import Path
from imdb_utils import utils

In [3]:
#  Constants
data_path = '../../data/bronze/imdb_raw_data/'

# Get the absolute path of where you are right now
src_path = Path.cwd()
project_src = src_path.parents[1]
print(f"Your project source root is: {project_src}")

Your project source root is: c:\Users\davib\Desktop\MSc_DataScience\DataWarehouse\lab_assignment\imbd


# Read raw sources

In [4]:
# Read raw data
title_basics = pd.read_csv(f'{data_path}/title.basics.tsv/data.tsv', sep='\t', na_values='\\N')
# title_basics = title_basics[title_basics['titleType'].isin(['tvSeries']) & title_basics['isAdult'] == 0]
# name_basics = pd.read_csv(f'{data_path}name.basics.tsv/data.tsv', sep='\t', dtype={'birthYear': 'Int64', 'deathYear': 'Int64'}, na_values='\\N')
title_principals = pd.read_csv(f'{data_path}/title.principals.tsv/data.tsv', sep='\t', na_values='\\N')

C:\Users\davib\AppData\Local\Temp\ipykernel_15292\1471240479.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  title_basics = pd.read_csv(f'{data_path}/title.basics.tsv/data.tsv', sep='\t', na_values='\\N')


# Dimensional tables

In [11]:
# Create a minimal filter dimension table
dim_filter = title_basics[['tconst', 'titleType', 'isAdult', 'startYear']].copy()
dim_filter = dim_filter[(dim_filter['titleType'] == 'tvSeries') & (dim_filter['isAdult'] == 0) & (dim_filter['startYear'] >= 2000)]

# Add nconst by joining with title_principals
dim_filter = dim_filter.merge(
    title_principals[['tconst', 'nconst']],
    on='tconst',
    how='inner'
).drop_duplicates()

# Reorder columns
dim_filter = dim_filter[['nconst', 'tconst', 'titleType', 'isAdult', 'startYear']]

print(f"dim_filter shape: {dim_filter.shape}")
print(f"Sample:\n{dim_filter.head()}")

# Write parquet file
dim_filter.to_parquet(f'{project_src}\\data\\silver\\dim_filter.parquet', compression='snappy', engine='pyarrow')

dim_filter shape: (800419, 5)
Sample:
      nconst     tconst titleType  isAdult  startYear
0  nm1014554  tt0111056  tvSeries      0.0     2000.0
1  nm1365013  tt0111056  tvSeries      0.0     2000.0
2  nm0410907  tt0111056  tvSeries      0.0     2000.0
3  nm1408401  tt0111056  tvSeries      0.0     2000.0
4  nm1020018  tt0111056  tvSeries      0.0     2000.0


In [4]:
dim_filter = pd.read_parquet(f'{project_src}\\data\\silver\\dim_filter.parquet')
dim_filter

,nconst,tconst,titleType,isAdult
0,nm0345300,tt0032557,tvSeries,0.0
1,nm0430460,tt0032557,tvSeries,0.0
2,nm0580585,tt0032557,tvSeries,0.0
3,nm0186608,tt0032557,tvSeries,0.0
4,nm0279963,tt0032557,tvSeries,0.0
...,...,...,...,...
1133546,nm8011539,tt9916678,tvSeries,0.0
1133547,nm10749212,tt9916678,tvSeries,0.0
1133548,nm9613196,tt9916678,tvSeries,0.0
1133549,nm10783601,tt9916678,tvSeries,0.0


## dim_profession | bridge_profession_group

In [5]:
# Create dim_profession from unique professions in primaryProfession
# Extract all unique professions from comma-separated primaryProfession values

# Sources:
# name_basics

# 2. Create dim_profession (Vectorized unique extraction)
unique_profs = (
    name_basics['primaryProfession']
    .str.split(',')
    .explode()
    .dropna()
    .unique()
)

dim_profession = pd.DataFrame({'profession_nm': sorted(unique_profs)})
dim_profession['profession_id'] = [f'prf_{i+1}' for i in range(len(dim_profession))]


# 3. Create bridge_profession_group (Vectorized expansion)
# Explode the professions column into individual rows
bridge_profession_group = (
    name_basics[['nconst', 'primaryProfession']]
    .dropna(subset=['primaryProfession'])
    .assign(profession_nm=lambda x: x['primaryProfession'].str.split(','))
    .explode('profession_nm')
)

# 4. Map the IDs using Merge (Replaces the dictionary lookup loop)
bridge_profession_group = (
    bridge_profession_group
    .reset_index()
    .merge(dim_profession.reset_index(), on='profession_nm')
    [['nconst', 'profession_id']]
)

# Create profession_group_id for each unique combination of professions per person
bridge_profession_group['profession_group_id'] = 'p_grp_' + (bridge_profession_group.groupby('nconst').ngroup() + 1).astype(str)

# Load dim_filter and apply inner join to keep only data in dim_filter
bridge_profession_group = bridge_profession_group.merge(
    dim_filter[['nconst']].drop_duplicates(),
    on='nconst',
    how='inner'
)
bridge_profession_group['weighting_factor_prf'] = 1 / bridge_profession_group.groupby('nconst')['profession_id'].transform('count')

# Write parquet files
bridge_profession_group.to_parquet(f'{project_src}\\data\\silver\\bridge_profession_group.parquet', compression='snappy', engine='pyarrow')
dim_profession.to_parquet(f'{project_src}\\data\\silver\\dim_profession.parquet', compression='snappy', engine='pyarrow')


print(f"\nbridge_profession_group shape (after filtering): {bridge_profession_group.shape}")
print(f"Sample bridge data:\n{bridge_profession_group.head(10)}")

del bridge_profession_group
del dim_profession


bridge_profession_group shape (after filtering): (780020, 4)
Sample bridge data:
      nconst profession_id profession_group_id  weighting_factor_prf
0  nm0000001        prf_35             p_grp_1              0.333333
1  nm0000001         prf_1             p_grp_1              0.333333
2  nm0000001        prf_25             p_grp_1              0.333333
3  nm0000002         prf_2             p_grp_2              0.500000
4  nm0000002        prf_35             p_grp_2              0.500000
5  nm0000003         prf_2             p_grp_3              0.333333
6  nm0000003        prf_35             p_grp_3              0.333333
7  nm0000003        prf_26             p_grp_3              0.333333
8  nm0000005        prf_41             p_grp_5              0.333333
9  nm0000005        prf_16             p_grp_5              0.333333


## dim_person

In [6]:
# Create dim_person from name_basics
# Load name_basics data

# Sources:
# name_basics
# bridge_profession_group
# dim_filter

bridge_profession_group = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_profession_group.parquet')
# dim_filter = pd.read_parquet(f'{project_src}\\data\\silver\\dim_filter.parquet')

# Select required columns for dim_person
dim_person = utils.select_cols(name_basics, ['nconst', 'primaryName', 'birthYear', 'deathYear'])

# Join with bridge_profssion_group to get profession_group_id
per_prfession = dim_person.merge(bridge_profession_group, on='nconst', how='left')
# Apply inner join with dim_filter to keep only participants in dim_filter
dim_person = dim_person.merge(
    dim_filter[['nconst']].drop_duplicates(),
    on='nconst',
    how='inner'
)

# Remove duplicates if any and set nconst as index
dim_person = per_prfession.drop_duplicates(subset=['nconst'])
dim_person = dim_person[['nconst','primaryName', 'birthYear', 'deathYear', 'profession_group_id']]

print(f"dim_person shape (after filtering): {dim_person.shape}")
# Write parquet files
dim_person.to_parquet(f'{project_src}\\data\\silver\\dim_person.parquet', compression='snappy', engine='pyarrow')

del dim_person

dim_person shape (after filtering): (10777803, 5)


## dim_roles

In [ ]:
import pandas as pd
import ast

# # --- Sources & Setup ---
cols_to_keep = ['sk_role', 'sk_person', 'sk_title', 'titleType', 'category', 'job', 'characters','char_rank']
# dim_filter = pd.read_parquet(f'{project_src}\\data\\silver\\dim_filter.parquet')

# # 1. Join title_principals with title_basics
dim_roles = (title_principals
             .merge(title_basics[['tconst', 'titleType']]
                    , on='tconst', how='left')
             .rename(columns={'nconst': 'sk_person', 'tconst': 'sk_title'}))

# 2. Apply inner join with dim_filter to keep only relevant participants
dim_roles = dim_roles.merge(
    dim_filter[['sk_person']].drop_duplicates(),
    on='sk_person',
    how='inner'
)

# 3. --- EXPLODE LOGIC ---
# Convert the string column '["Char1", "Char2"]' into a Python list
def parse_char_list(x):
    if pd.isna(x) or x == r'\N':
        return [None]
    try:
        # ast.literal_eval safely evaluates a string into a list
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return [x]

dim_roles['characters'] = dim_roles['characters'].apply(parse_char_list)

# Explode the list into separate rows
dim_roles = dim_roles.explode('characters')

# 4. Create a unique role_id 
# Since we exploded, we add a cumulative count to the ID to keep it unique per character
dim_roles['char_rank'] = dim_roles.groupby(['sk_person', 'sk_title', 'ordering']).cumcount() + 1
dim_roles['sk_role'] = (
    dim_roles['sk_person'] + '_' + 
    dim_roles['sk_title'] + '_' + 
    dim_roles['ordering'].astype(str) + '_' + 
    dim_roles['char_rank'].astype(str)
)

# 5. Select and reorder columns
dim_roles = utils.select_cols(dim_roles, cols_to_keep)

# 6. Remove duplicates based on the new granular role_id
dim_roles = dim_roles.drop_duplicates(subset=['sk_role'])

# 7. Final filtering with dim_filter
dim_roles = dim_roles.merge(
    dim_filter[['sk_person', 'sk_title']].drop_duplicates(),
    on=['sk_person', 'sk_title'],
    how='inner'
)

# 8. Rename to DDL standard
dim_roles_mapping = {
    'role_id': 'sk_role',
    'nconst': 'sk_person',
    'tconst': 'sk_title',
    'titleType': 'titleType',
    'category': 'category',
    'job': 'job',
    'characters': 'characters',
    'char_rank': 'char_rank'
}

dim_roles = dim_roles.rename(columns=dim_roles_mapping)

print(f"dim_roles shape (after exploding and filtering): {dim_roles.shape}")

# Write parquet files
dim_roles.to_parquet(f'{project_src}\\data\\silver\\dim_roles.parquet', compression='snappy', engine='pyarrow')

# del dim_roles

dim_roles shape (after exploding and filtering): (906524, 8)


In [5]:
dim_roles = pd.read_parquet(f'{project_src}\\data\\silver\\dim_roles.parquet')
# dim_roles.to_parquet(f'{project_src}\\data\\silver\\dim_roles.parquet', compression='snappy', engine='pyarrow')
dim_roles

,sk_role,sk_person,sk_title,titleType,category,job,characters,char_rank
0,nm1014554_tt0111056_10_1,nm1014554,tt0111056,tvSeries,actress,None,Yaone,1
1,nm1365013_tt0111056_1_1,nm1365013,tt0111056,tvSeries,actor,None,Cho Hakkai,1
2,nm0410907_tt0111056_2_1,nm0410907,tt0111056,tvSeries,actor,None,Cho Hakkai,1
3,nm1408401_tt0111056_3_1,nm1408401,tt0111056,tvSeries,actor,None,Demon,1
4,nm1020018_tt0111056_4_1,nm1020018,tt0111056,tvSeries,actress,None,Sanbutsushin,1
...,...,...,...,...,...,...,...,...
906519,nm9613196_tt9916678_7_2,nm9613196,tt9916678,tvSeries,actress,None,Lícia,2
906520,nm10783601_tt9916678_8_1,nm10783601,tt9916678,tvSeries,actor,None,Lucas,1
906521,nm10783601_tt9916678_8_2,nm10783601,tt9916678,tvSeries,actor,None,Rafael,2
906522,nm5828671_tt9916678_9_1,nm5828671,tt9916678,tvSeries,actress,None,Carla,1


## dim_genre | bridge_genre_group

In [ ]:
# 1. Load Data
# Sources:
# title_basics
# dim_filter

dim_filter = pd.read_parquet(f'{project_src}\\data\\silver\\dim_filter.parquet')

# 2. Create dim_genres (The Dimension Table)
# Extract unique genres using vectorized split and unique()
unique_genres = (
    title_basics['genres']
    .str.split(',')
    .explode()
    .dropna()
    .unique()
)

dim_genres = pd.DataFrame({'genre_nm': sorted(unique_genres)})
dim_genres['sk_genre'] = [f'gen_{i+1}' for i in range(len(dim_genres))]

# 3. Create bridge_genres (The Bridge Table)
# Instead of a dictionary lookup, we use explode and then merge with dim_genres
bridge_genres = (
    title_basics[['tconst', 'genres']].rename(columns={'tconst': 'sk_title'})
    .dropna(subset=['genres'])
    .assign(genre_nm=lambda x: x['genres'].str.split(','))
    .explode('genre_nm')
)

# Merge to get the genre_id (vectorized replacement for the dictionary lookup)
bridge_genres = (
    bridge_genres.reset_index()
    .merge(dim_genres.reset_index(), on='genre_nm')
    [['sk_title', 'sk_genre']]
)

# 4. Create genre_group_id
# This stays vectorized as before
bridge_genres['sk_genre_group'] = (
    'g_grp_' + 
    (bridge_genres.groupby('sk_title').ngroup() + 1).astype(str)
)
# Calculate the weight as 1 divided by the number of genres for that specific title
bridge_genres['weighting_factor_gen'] = 1 / bridge_genres.groupby('sk_title')['sk_genre'].transform('count')

# Apply inner join with dim_filter to keep only genres for titles in dim_filter
bridge_genres = bridge_genres.merge(
    dim_filter[['tconst']].drop_duplicates(),
    on='tconst',
    how='inner'
)

print(f"dim_genres shape: {dim_genres.shape}")
print(f"bridge_genres shape (after filtering): {bridge_genres.shape}")

# Write parquet files
# dim_genres.to_parquet(f'{project_src}\\data\\silver\\dim_genres.parquet', compression='snappy', engine='pyarrow')
# bridge_genres.to_parquet(f'{project_src}\\data\\silver\\bridge_genres.parquet', compression='snappy', engine='pyarrow')

del dim_genres
del bridge_genres

dim_genres shape: (28, 2)
bridge_genres shape (after filtering): (225747, 4)


## dim_title_basic 

In [6]:
# Sources:
# title_basics
# bridge_genres
# dim_filter

bridge_genres = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_genres.parquet')
# Select required columns for dim_title_basic
# dim_title_basic = title_basics[['tconst', 'titleType','primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes']].copy()
dim_title_basic = utils.select_cols(title_basics, ['tconst', 'titleType','primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes'])

# Apply inner join with dim_filter to keep only titles in dim_filter
dim_title_basic = dim_title_basic.merge(
    dim_filter[['tconst']].drop_duplicates(),
    on='tconst',
    how='inner'
)

# Join with bridge_profssion_group to get profession_group_id
title_genre = dim_title_basic.join(bridge_genres.set_index('tconst'), on='tconst', how='left')
# title_genre = title_genre[['tconst', 'titleType','primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genre_group_id']].copy()
title_genre = utils.select_cols(title_genre, ['tconst', 'titleType','primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genre_group_id'])
# Remove duplicates if any and set tconst as index
dim_title_basic = title_genre.drop_duplicates(subset=['tconst'])

# Convert to numeric, forcing errors to NaN
dim_title_basic['runtimeMinutes'] = pd.to_numeric(dim_title_basic['runtimeMinutes'], errors='coerce')

# dim_title_basic = dim_title_basic[['tconst', 'titleType','primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genre_group_id']]
print(f"dim_title_basic shape (after filtering): {dim_title_basic.shape}")

#Write parquet files
dim_title_basic.to_parquet(f'{project_src}\\data\\silver\\dim_title_basic.parquet', compression='snappy', engine='pyarrow')

del dim_title_basic

C:\Users\davib\AppData\Local\Temp\ipykernel_23240\1221509368.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dim_title_basic['runtimeMinutes'] = pd.to_numeric(dim_title_basic['runtimeMinutes'], errors='coerce')


dim_title_basic shape (after filtering): (175011, 9)


In [ ]:
dim_title_basic  = pd.read_parquet(f'{project_src}\\data\\silver\\dim_title_basic .parquet')
# Define the mapping dictionary for the bridge table
dim_title_basic  = {
    'tconst': 'sk_title',
    'genre_group_id': 'sk_genre_group'
}

# Rename the columns in the bridge_kwn_titles dataframe
bridge_kwn_titles = bridge_kwn_titles.rename(columns=bridge_column_mapping)

##  bridge_kt_group

In [14]:
# 1. Load data (Keeping your existing logic)
# Note: Using na_values='\\N' as discussed previously is perfect here

# Sources:
# title_basics
# name_basics 

name_basics = pd.read_csv(f'{data_path}name.basics.tsv/data.tsv', sep='\t', dtype={'birthYear': 'Int64', 'deathYear': 'Int64'}, na_values='\\N')

# 2. Vectorized Expansion (Replacing the two loops)
# We select only the columns we need, split the strings into lists, and "explode" them
# 1. Vectorized Expansion (Replacing the two loops)
# We select only the columns we need, split the strings into lists, and "explode" them
bridge_kwn_titles = (
    name_basics[['nconst', 'knownForTitles']]
    .dropna(subset=['knownForTitles'])
    .assign(tconst=lambda x: x['knownForTitles'].str.split(','))
    .explode('tconst')
    [['tconst', 'nconst']]
    .rename(columns={'tconst': 'sk_title', 'nconst': 'sk_person'})  # Reorder to match your desired output
)

# Apply inner join with dim_filter to keep only known titles for participants and titles in dim_filter
bridge_kwn_titles = bridge_kwn_titles.merge(
    dim_filter[['sk_person', 'sk_title']].drop_duplicates(),
    on=['sk_person', 'sk_title'],
    how='inner'
)

# 2. Vectorized ID Generation
# ngroup() is already vectorized, so we just keep this logic
bridge_kwn_titles['sk_kwn_title_group'] = (
    'kwn_t_grp_' + 
    (bridge_kwn_titles.groupby('sk_title').ngroup() + 1).astype(str)
)
bridge_kwn_titles['weighting_factor_grp'] = 1 / bridge_kwn_titles.groupby('sk_person')['sk_title'].transform('count')


print(f"bridge_kwn_titles shape (after filtering): {bridge_kwn_titles.shape}")
# bridge_kwn_titles.to_parquet(f'{project_src}\\data\\silver\\bridge_kwn_titles.parquet', compression='snappy', engine='pyarrow')

# del bridge_kwn_titles

bridge_kwn_titles shape (after filtering): (314968, 4)


In [4]:
bridge_kwn_titles = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_kwn_titles.parquet')

bridge_kwn_titles[bridge_kwn_titles['sk_person'] == 'nm9993327']

,sk_title,sk_person,sk_kwn_title_group,weighting_factor_grp
314958,tt8754282,nm9993327,kwn_t_grp_88840,0.25
314959,tt8759944,nm9993327,kwn_t_grp_88866,0.25
314960,tt8754200,nm9993327,kwn_t_grp_88839,0.25
314961,tt8760492,nm9993327,kwn_t_grp_88873,0.25


# Factual tables

## participations

In [20]:
# Read dims
dim_person = pd.read_parquet(f'{project_src}\\data\\silver\\dim_person.parquet')
dim_roles = pd.read_parquet(f'{project_src}\\data\\silver\\dim_roles.parquet')
dim_title_basic = pd.read_parquet(f'{project_src}\\data\\silver\\dim_title_basic.parquet')
bridge_kwn_titles = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_kwn_titles.parquet')
dim_filter = pd.read_parquet(f'{project_src}\\data\\silver\\dim_filter.parquet')

# Convert repetitive strings to categories
dim_roles['category'] = dim_roles['category'].astype('category')
dim_roles['job'] = dim_roles['job'].astype('category')
dim_title_basic['titleType'] = dim_title_basic['titleType'].astype('category')

# If you are using Pandas 2.0+, convert IDs and text to PyArrow strings
# This is drastically more memory efficient than standard 'object' strings
# dim_roles['tconst'] = dim_roles['tconst'].astype('string[pyarrow]')

In [15]:
dim_title_basic

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genre_group_id
0,tt0032557,tvSeries,The Green Archer,The Green Archer,0.0,1940.0,NaN,285.0,g_grp_29229
1,tt0035803,tvSeries,The German Weekly Review,Die Deutsche Wochenschau,0.0,1940.0,1945.0,12.0,g_grp_32221
2,tt0038276,tvSeries,You Are an Artist,You Are an Artist,0.0,1946.0,1955.0,15.0,None
3,tt0039120,tvSeries,Americana,Americana,0.0,1947.0,1949.0,30.0,g_grp_35265
4,tt0039121,tvSeries,Birthday Party,Birthday Party,0.0,1947.0,1949.0,30.0,g_grp_35266
...,...,...,...,...,...,...,...,...,...
175006,tt9916206,tvSeries,Nojor,Nojor,0.0,2019.0,NaN,20.0,g_grp_7095209
175007,tt9916216,tvSeries,Kalyanam Mudhal Kadhal Varai,Kalyanam Mudhal Kadhal Varai,0.0,2014.0,2017.0,22.0,g_grp_7095213
175008,tt9916218,tvSeries,Lost in Food,Lost in Food,0.0,2016.0,2017.0,NaN,g_grp_7095214
175009,tt9916380,tvSeries,Meie aasta Aafrikas,Meie aasta Aafrikas,0.0,2019.0,NaN,43.0,g_grp_7095283


In [ ]:
import gc

# Merge 1: Titles and Roles
participations_pers = (
    dim_title_basic[['tconst','titleType', 'primaryTitle', 'genre_group_id', 'runtimeMinutes']]
                       .rename(columns={'tconst': 'sk_title'})
                       .merge(
    dim_roles[['sk_role','sk_title', 'sk_person', 'category', 'job', 'characters']],
    on='sk_title', 
    how='left'
    )
)

# Optional: If you no longer need the source dataframes in this script, delete them to free RAM
del dim_roles
del dim_title_basic
gc.collect() # Force Python to clear unreferenced memory immediately

# Merge 2: Add Person data
participations_pers = participations_pers.merge(
    dim_person[['sk_person', 'primaryName', 'sk_profession_group']], 
    on='sk_person', 
    how='left'
)
del dim_person
gc.collect()

# Merge 3: Add Bridge
# Ensure bridge_kwn_titles is also pre-filtered to ONLY the columns you need before merging
participations_pers = participations_pers.merge(
    bridge_kwn_titles, 
    on=['sk_title', 'sk_person'], 
    how='left'
)
del bridge_kwn_titles
gc.collect()

# participations_pers = participations_pers[['nconst', 'primaryName', 'tconst', 'titleType', 'primaryTitle', 'runtimeMinutes', 'genre_group_id', 'category', 'job', 'characters', 'profession_group_id', 'kwn_title_group_id']].copy()
participations_pers = utils.select_cols(participations_pers, ['sk_person', 'primaryName', 'sk_title', 'titleType', 'primaryTitle', 'runtimeMinutes', 'genre_group_id', 'sk_role', 'category', 'job', 'characters', 'sk_profession_group', 'sk_kwn_title_group'])
participations_pers['participation_count'] = 1

print(f"participations_pers.shape: {participations_pers.shape}")

participations_pers = participations_pers.merge(
    dim_filter[['sk_person', 'sk_title']].drop_duplicates(),
    on=['sk_person', 'sk_title'],
    how='inner'
)

print(f"participations_pers.shape after filtering: {participations_pers.shape}")

# #Write parquet file
# participations_pers.to_parquet(f'{project_src}\\data\\silver\\participations_pers.parquet', compression='snappy', engine='pyarrow')

# del participations_pers

participations_pers.shape: (956019, 14)
participations_pers.shape after filtering: (906524, 14)


In [23]:
# del participations_pers
# participations_pers = pd.read_parquet(f'{project_src}\\data\\silver\\participations_pers.parquet')
participations_pers
# tt0032557

,sk_person,primaryName,sk_title,titleType,primaryTitle,runtimeMinutes,genre_group_id,sk_role,category,job,characters,sk_profession_group,sk_kwn_title_group,participation_count
0,nm1014554,Shelley Calene-Black,tt0111056,tvSeries,Gensomaden Saiyuki,23.0,g_grp_100833,nm1014554_tt0111056_10_1,actress,NaN,Yaone,p_grp_1007880,NaN,1
1,nm1365013,Braden Hunt,tt0111056,tvSeries,Gensomaden Saiyuki,23.0,g_grp_100833,nm1365013_tt0111056_1_1,actor,NaN,Cho Hakkai,p_grp_2545327,kwn_t_grp_1,1
2,nm0410907,Akira Ishida,tt0111056,tvSeries,Gensomaden Saiyuki,23.0,g_grp_100833,nm0410907_tt0111056_2_1,actor,NaN,Cho Hakkai,p_grp_380497,NaN,1
3,nm1408401,John Area,tt0111056,tvSeries,Gensomaden Saiyuki,23.0,g_grp_100833,nm1408401_tt0111056_3_1,actor,NaN,Demon,p_grp_2581415,kwn_t_grp_1,1
4,nm1020018,Christine M. Auten,tt0111056,tvSeries,Gensomaden Saiyuki,23.0,g_grp_100833,nm1020018_tt0111056_4_1,actress,NaN,Sanbutsushin,p_grp_1046786,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
906519,nm9613196,Antonella Mattos,tt9916678,tvSeries,Acelerados,NaN,g_grp_7095378,nm9613196_tt9916678_7_2,actress,NaN,Lícia,p_grp_8256203,kwn_t_grp_94006,1
906520,nm10783601,Miguel Schmidt,tt9916678,tvSeries,Acelerados,NaN,g_grp_7095378,nm10783601_tt9916678_8_1,actor,NaN,Lucas,p_grp_1419278,kwn_t_grp_94006,1
906521,nm10783601,Miguel Schmidt,tt9916678,tvSeries,Acelerados,NaN,g_grp_7095378,nm10783601_tt9916678_8_2,actor,NaN,Rafael,p_grp_1419278,kwn_t_grp_94006,1
906522,nm5828671,Maria Clara Vicente,tt9916678,tvSeries,Acelerados,NaN,g_grp_7095378,nm5828671_tt9916678_9_1,actress,NaN,Carla,p_grp_5906541,kwn_t_grp_94006,1
